# Week 2 Step 10: Satisfaction on open tickets

The three core models are saved. This extra model predicts `customer_satisfaction_score` for tickets that are still open.

Closed and Resolved tickets have a score. Pending and Escalated tickets never do. I train on the scored tickets, using issue metadata plus the ticket text. I do not use `resolution_time_hours` or `ticket_status`: those are missing or different for open tickets, so they would not be there at prediction time.

In [2]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RANDOM_STATE = 42
tickets = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "support_tickets_clean.csv")

scored = tickets.loc[tickets["customer_satisfaction_score"].notna()].copy()
open_tickets = tickets.loc[tickets["customer_satisfaction_score"].isna()].copy()

print("all tickets:", len(tickets))
print("scored (Resolved/Closed):", len(scored), scored["ticket_status"].value_counts().to_dict())
print("open (Pending/Escalated):", len(open_tickets), open_tickets["ticket_status"].value_counts().to_dict())
print("CSAT mean on scored:", round(scored["customer_satisfaction_score"].mean(), 2))

all tickets: 6120
scored (Resolved/Closed): 4592 {'Resolved': 3390, 'Closed': 1202}
open (Pending/Escalated): 1528 {'Pending': 888, 'Escalated': 640}
CSAT mean on scored: 3.26


In [3]:
cat_cols = ["issue_category", "issue_subcategory", "priority"]
text_col = "ticket_description"
feature_cols = cat_cols + [text_col]

X = scored[feature_cols]
y = scored["customer_satisfaction_score"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

meta_only = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)
meta_and_text = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        (
            "text",
            TfidfVectorizer(max_features=400, min_df=2, stop_words="english"),
            text_col,
        ),
    ]
)

candidates = {
    "ridge_metadata": Pipeline([("prep", meta_only), ("model", Ridge(random_state=RANDOM_STATE))]),
    "ridge_metadata_and_text": Pipeline(
        [("prep", meta_and_text), ("model", Ridge(random_state=RANDOM_STATE))]
    ),
}

rows = []
fitted = {}
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    rows.append(
        {
            "model": name,
            "mae": mean_absolute_error(y_test, pred),
            "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
            "r2": r2_score(y_test, pred),
        }
    )

results = pd.DataFrame(rows).sort_values("mae")
display(results.round(3))

best_name = results.iloc[0]["model"]
best_pipe = fitted[best_name]
best_mae = results.iloc[0]["mae"]
print("model I am using for open tickets:", best_name)
print(f"On scored test tickets, predictions are off by about {best_mae:.2f} CSAT points on average.")

,model,mae,rmse,r2
0,ridge_metadata,0.869,1.066,0.056
1,ridge_metadata_and_text,0.874,1.068,0.054


model I am using for open tickets: ridge_metadata
On scored test tickets, predictions are off by about 0.87 CSAT points on average.


In [4]:
open_tickets = open_tickets.copy()
open_tickets["predicted_csat"] = best_pipe.predict(open_tickets[feature_cols]).clip(1, 5)

print("predicted CSAT on open tickets:")
print(open_tickets["predicted_csat"].describe().round(2))
display(
    open_tickets.groupby("ticket_status")["predicted_csat"]
    .agg(["count", "mean", "median"])
    .round(2)
)
display(
    open_tickets[
        ["ticket_id", "ticket_status", "issue_category", "priority", "predicted_csat"]
    ].head()
)

out_path = PROJECT_ROOT / "data" / "processed" / "open_ticket_csat_predictions.csv"
open_tickets.to_csv(out_path, index=False)
joblib.dump(best_pipe, PROJECT_ROOT / "models" / "satisfaction_model.pkl")
print("saved", out_path.name, "and satisfaction_model.pkl")

predicted CSAT on open tickets:
count    1528.00
mean        3.27
std         0.27
min         2.91
25%         2.99
50%         3.38
75%         3.52
max         3.65
Name: predicted_csat, dtype: float64


,count,mean,median
ticket_status,,,
Escalated,640,3.27,3.41
Pending,888,3.26,3.36


,ticket_id,ticket_status,issue_category,priority,predicted_csat
3,TCK0000004,Escalated,Account Access,Low,3.582371
9,TCK0000010,Escalated,Streaming Quality,Medium,2.938979
16,TCK0000017,Escalated,Billing & Payments,Medium,2.928147
21,TCK0000022,Pending,Streaming Quality,High,3.123544
28,TCK0000029,Escalated,Subscription,Low,3.594672


saved open_ticket_csat_predictions.csv and satisfaction_model.pkl


This is a bonus model. CSAT only exists after a ticket is closed, so I cannot check the open-ticket predictions against a true score. If metadata-plus-text is barely better than metadata only, the descriptions are not adding much and I would treat the scores as a rough queue flag, not as a precise CSAT forecast.